# 03: Study Design Classification — Validation

Validates Phase 2B output: `class.study_design` (L1/L2/L4/L5) and `class.innovative_features` (L3).

**Checks:**
- Distribution of each classification level
- Innovative feature counts by type
- Spot-check examples per feature type for precision

In [16]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from config.settings import get_duckdb_connection

conn = get_duckdb_connection()
print("Connected to DuckDB")

Connected to DuckDB


## 1. Study Design Classification Overview

In [17]:
total = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
print(f"Total studies classified: {total:,}")

Total studies classified: 118,764


### L1: Study Type

In [18]:
conn.execute("""
    SELECT study_type, COUNT(*) AS count
    FROM class.study_design
    GROUP BY study_type
    ORDER BY count DESC
""").fetchdf()

,study_type,count
0,INTERVENTIONAL,87581
1,OBSERVATIONAL,30931
2,EXPANDED_ACCESS,252


### L2: Design Architecture

In [19]:
conn.execute("""
    SELECT design_architecture, COUNT(*) AS count
    FROM class.study_design
    WHERE design_architecture IS NOT NULL
    GROUP BY design_architecture
    ORDER BY count DESC
""").fetchdf()

,design_architecture,count
0,Parallel RCT,47427
1,Single-Arm,24339
2,Cohort,20441
3,Non-Randomized Parallel,5045
4,Crossover RCT,3949
5,Case-Control,3405
6,Other Observational,3381
7,Case-Only,3019
8,Non-Randomized Sequential,2296
9,Sequential RCT,1403


### L4: Blinding Level

In [20]:
conn.execute("""
    SELECT blinding_level, COUNT(*) AS count
    FROM class.study_design
    WHERE blinding_level IS NOT NULL
    GROUP BY blinding_level
    ORDER BY count DESC
""").fetchdf()

,blinding_level,count
0,Open Label,52955
1,Single Blind,13300
2,Double Blind,9329
3,Quadruple Blind,6907
4,Triple Blind,5063


### L5: Purpose

In [21]:
conn.execute("""
    SELECT purpose, COUNT(*) AS count
    FROM class.study_design
    WHERE purpose IS NOT NULL
    GROUP BY purpose
    ORDER BY count DESC
""").fetchdf()

,purpose,count
0,TREATMENT,57207
1,PREVENTION,8315
2,OTHER,5398
3,SUPPORTIVE_CARE,5395
4,DIAGNOSTIC,3696
5,BASIC_SCIENCE,3594
6,HEALTH_SERVICES_RESEARCH,2649
7,SCREENING,934
8,DEVICE_FEASIBILITY,388


## 2. Innovative Features (L3)

In [22]:
total_features = conn.execute("SELECT COUNT(*) FROM class.innovative_features").fetchone()[0]
total_studies = conn.execute("SELECT COUNT(DISTINCT nct_id) FROM class.innovative_features").fetchone()[0]
all_studies = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
pct = round(100 * total_studies / all_studies, 1)
print(f"Total feature rows: {total_features:,}")
print(f"Studies with at least one innovative feature: {total_studies:,} ({pct}%)")

Total feature rows: 7,014
Studies with at least one innovative feature: 4,680 (3.9%)


In [23]:
conn.execute("""
    SELECT feature_type, COUNT(DISTINCT nct_id) AS study_count
    FROM class.innovative_features
    GROUP BY feature_type
    ORDER BY study_count DESC
""").fetchdf()

,feature_type,study_count
0,adaptive,2519
1,pragmatic,1194
2,platform,268
3,bayesian,256
4,SMART,222
5,umbrella,182
6,master protocol,167
7,basket,151
8,seamless,126
9,N-of-1,49


### Source field breakdown

In [24]:
conn.execute("""
    SELECT feature_type, source_field, COUNT(*) AS count
    FROM class.innovative_features
    GROUP BY feature_type, source_field
    ORDER BY feature_type, count DESC
""").fetchdf()

,feature_type,source_field,count
0,N-of-1,description,32
1,N-of-1,official_title,30
2,N-of-1,keyword,14
3,N-of-1,brief_title,13
4,SMART,description,190
5,SMART,official_title,90
6,SMART,brief_title,54
7,SMART,keyword,29
8,adaptive,description,1938
9,adaptive,official_title,828


### Co-occurrence: studies with multiple features

In [25]:
conn.execute("""
    SELECT feature_count, COUNT(*) AS studies
    FROM (
        SELECT nct_id, COUNT(DISTINCT feature_type) AS feature_count
        FROM class.innovative_features
        GROUP BY nct_id
    )
    GROUP BY feature_count
    ORDER BY feature_count
""").fetchdf()

,feature_count,studies
0,1,4309
1,2,297
2,3,58
3,4,14
4,5,2


## 3. AI/ML Mentions (Phase 2B.1)

`class.ai_mentions` tracks studies that mention AI, machine learning, deep learning, NLP, etc. in their free-text fields. Independent from `innovative_features` — a study can reference AI/ML without having an AI-augmented *design* (e.g. using ML only for analysis).

In [ ]:
ai_total = conn.execute("SELECT COUNT(*) FROM class.ai_mentions").fetchone()[0]
ai_studies = conn.execute("SELECT COUNT(DISTINCT nct_id) FROM class.ai_mentions").fetchone()[0]
all_studies = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
ai_pct = round(100 * ai_studies / all_studies, 1)
print(f"AI/ML mention rows:        {ai_total:,}")
print(f"Studies with an AI mention: {ai_studies:,} ({ai_pct}%)")

In [ ]:
conn.execute("""
    SELECT ai_term, COUNT(DISTINCT nct_id) AS studies
    FROM class.ai_mentions
    GROUP BY ai_term
    ORDER BY studies DESC
""").fetchdf()

## 4. Spot-Checks

Random sample of 10 studies per feature type for manual precision verification.

In [26]:
feature_types = conn.execute("""
    SELECT DISTINCT feature_type FROM class.innovative_features ORDER BY feature_type
""").fetchall()

for (ft,) in feature_types:
    print(f"\n{'='*80}")
    print(f"Feature: {ft}")
    print(f"{'='*80}")
    samples = conn.execute(f"""
        SELECT f.nct_id, f.source_field, f.matched_text, s.brief_title
        FROM class.innovative_features f
        JOIN raw.studies s ON f.nct_id = s.nct_id
        WHERE f.feature_type = '{ft}'
        ORDER BY random()
        LIMIT 10
    """).fetchdf()
    display(samples)


Feature: N-of-1


,nct_id,source_field,matched_text,brief_title
0,NCT07343024,brief_title,N-of-1,Methods of Identifying Effective Off-Guideline...
1,NCT07377032,description,n-of-1,TAP-GRIN: Interventional Study on Patients Wit...
2,NCT06016517,official_title,N-of-1,Application of the Personalized N-of-1 Trial D...
3,NCT07155291,official_title,N-of-1,Menstrual Pain Intervention Among Students
4,NCT05776056,description,N-of-1,Methylphenidate for the Treatment of PTSD With...
5,NCT07087587,official_title,N-of-1,Sleep Apnea Triggers of Atrial Fibrillation: N...
6,NCT06749964,keyword,n-of-1,An N-of-1 Trial of an Internet-delivered CBT P...
7,NCT07344909,official_title,N-of-1,Burning Mouth Syndrome: Effects of Occlusal Sp...
8,NCT04580368,description,N-of-1,Testing Drug Efficacy in Cystic Fibrosis Throu...
9,NCT04580368,brief_title,N-of-1,Testing Drug Efficacy in Cystic Fibrosis Throu...



Feature: SMART


,nct_id,source_field,matched_text,brief_title
0,NCT06363019,description,SMART,Supporting At-Risk Mothers Across Perinatal Pe...
1,NCT04978584,description,SMART,"Rituximab, Lenalidomide, Acalabrutinib, Tafasi..."
2,NCT07376135,description,SMART,LETHE-AT: a Personalized Multidomain Lifestyle...
3,NCT06886789,official_title,SMART,SMART to Optimize an Intervention to Maintain ...
4,NCT06890676,description,SMART,Extending Dental Care to Nursing Home Resident...
5,NCT06813235,official_title,Sequential Multiple Assignment Randomized,Dream2Heal: An Adaptive Stepped-care Intervent...
6,NCT05619185,brief_title,SMART,A SMART Evaluation of an Adaptive Web-based AU...
7,NCT07136610,brief_title,SMART,Standardized Application of Feeding Evaluation...
8,NCT06919952,description,SMART,Comparing a Workplace Resilience and a Physica...
9,NCT05185947,keyword,SMART,Study of Intravenous and Intraperitoneal Pacli...



Feature: adaptive


...

,nct_id,source_field,matched_text,brief_title
0,NCT05669833,official_title,Pragmatic,Guselkumab vs Golimumab in PsA TNF Inadequate ...
1,NCT05871008,description,pragmatic,Integrated Actionable Aging Assessment for Can...
2,NCT06964867,description,pragmatic,Effectiveness of Multimedia Health Education t...
3,NCT07073794,official_title,Pragmatic,Evaluating In Home Cancer Therapy Versus In Cl...
4,NCT07424690,description,pragmatic,Catheter Ablation vs Conservative Care in Elde...
5,NCT06323824,description,pragmatic,Office-based Methadone Versus Buprenorphine to...
6,NCT06110273,description,pragmatic,Fit for Duty: mHealth Intervention for Weight ...
7,NCT06621355,description,pragmatic,HemoNIRS: a Paradigm Shift in the Process of M...
8,NCT06979830,description,pragmatic,The Effects of Yoga Exercises in Overweight an...
9,NCT06843603,description,pragmatic,Barriers/Facilitators and Care Coordination of...



Feature: seamless


,nct_id,source_field,matched_text,brief_title
0,NCT05890729,official_title,Seamless,A Study of XTMAB-16 in Patients With Pulmonary...
1,NCT06817941,description,seamless,Speech and Arm Combined Exergame
2,NCT06051695,official_title,Seamless,A Study to Evaluate the Safety and Efficacy of...
3,NCT07458698,description,seamless,Reaching Calm: A Digital Intervention to Preve...
4,NCT07120347,description,seamless,Remote Sensing for ADRD-Specific Activities Id...
5,NCT05502341,official_title,Seamless,Study to Compare Bictegravir/Lenacapavir Versu...
6,NCT07430813,description,seamless,Drone Delivery of Automated External Defibrill...
7,NCT07126288,description,seamless,Evaluating the Efficacy and Safety of GB08 Inj...
8,NCT06789601,description,seamless,Evaluating Informatics-assisted Immune-related...
9,NCT06630260,description,seamless,5G-RUBY: Avutometinib and Defactinib in Malign...



Feature: umbrella


,nct_id,source_field,matched_text,brief_title
0,NCT05722886,official_title,Umbrella,DETERMINE (Determining Extended Therapeutic In...
1,NCT06780098,official_title,Umbrella,Substudy 01I: A Study of Investigational Agent...
2,NCT05781763,description,umbrella,High-intensity Training in Patients With Spond...
3,NCT07303582,description,umbrella,Individualised Cryoneurolysis to Treat Pain in...
4,NCT04298021,brief_title,Umbrella,DDR-Umbrella Study of DDR Targeting Agents in ...
5,NCT04298021,official_title,Umbrella,DDR-Umbrella Study of DDR Targeting Agents in ...
6,NCT03753412,description,umbrella,Recovery From ICUAW Following Severe Respirato...
7,NCT04298021,description,umbrella,DDR-Umbrella Study of DDR Targeting Agents in ...
8,NCT04929223,official_title,Umbrella,A Study Evaluating the Safety and Efficacy of ...
9,NCT05573555,keyword,Umbrella,TACTIVE-U: A Study to Learn About the Study Me...


In [27]:
conn.close()